In [ ]:
import io
import re
import requests
import pandas as pd
pd.set_option('display.max_columns', None)
import pyarrow as pa
import pyarrow.parquet as pq
from bs4 import BeautifulSoup
from google.cloud import storage
import source.helpers.gcs_helper as gcs
from source.helpers.seventeen_lands_dtypes import COLUMN_TYPES
import source.helpers.streaming_data_to_cloud as stream
import gzip
import json
import tempfile
from urllib.parse import urlparse

# --- CONFIGURATION ---
GCS_BUCKET_NAME = "mtg-open-source-data"
DATASETS_PAGE_URL = "https://17lands-public.s3.amazonaws.com/analysis_data"
CHUNK_SIZE = 100000  # Number of rows processed at a time (adjust based on RAM)
gcs_client = gcs.get_gcs_client()

# 1. Fetch the bulk-data metadata catalog from Scryfall
meta_url = "https://api.scryfall.com/bulk-data"
headers = {
    "User-Agent": "MyMTGApp/1.0",
    "Accept": "application/json",
}

response = requests.get(meta_url, headers=headers, verify=False)
response.raise_for_status()
bulk_metadata = response.json()["data"]


# 2. Find the entry for 'default-cards' (contains every unique card object in English/printed language)
# You can also use 'all-cards' if you need every single printing in every language.
target_type = "default_cards"
download_uri = None

for item in bulk_metadata: 
    if item["type"] == target_type:
        download_uri = item["jsonl_download_uri"]
        break

c:\Dev\mtg\.venv\Lib\site-packages\urllib3\connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.scryfall.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [3]:
from google.cloud import bigquery

bq_client = bigquery.Client()
query = """
        SELECT
            `set`,
            `set_id`,
            `set_name`, 
            MIN(`released_at`) AS `release_date`
        FROM
            `mtg-analytics-514317.open_source_data.scryfall_default_cards`
        GROUP BY ALL
    """

# Configure parameters for the query
job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("filter_val", "STRING", "example_value"),
        bigquery.ScalarQueryParameter("limit_val", "INT64", 1000),
    ]
)

# Run the query
query_job = bq_client.query(query, job_config=job_config)

# Option A: Convert the results directly into a Pandas DataFrame
df = query_job.to_dataframe()

c:\Dev\mtg\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [4]:
df.head()

,set,set_id,set_name,release_date
0,blb,a2f58272-bba6-439d-871e-7a46686ac018,Bloomburrow,2024-08-02 00:00:00+00:00
1,tsp,c1d109bc-ffd8-428f-8d7d-3f8d7e648046,Time Spiral,2006-10-06 00:00:00+00:00
2,zen,eb16a2bd-a218-4e4e-8339-4aa1afc0c8d2,Zendikar,2009-10-02 00:00:00+00:00
3,tmm2,f7aa47c6-c1e2-4de5-9a68-4406d84bd6bb,Modern Masters 2015 Tokens,2015-05-22 00:00:00+00:00
4,xln,fe0dad85-54bc-4151-9200-d68da84dd0f2,Ixalan,2017-09-29 00:00:00+00:00


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1066 entries, 0 to 1065
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   set           1066 non-null   str                
 1   set_id        1066 non-null   str                
 2   set_name      1066 non-null   str                
 3   release_date  1066 non-null   datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), str(3)
memory usage: 98.4 KB


In [8]:
from datetime import datetime, timezone
min_date_sev_lands = datetime.fromisoformat('2021-04-04 00:00:00').replace(
    tzinfo=timezone.utc
)
df[df['release_date'] > min_date_sev_lands]

,set,set_id,set_name,release_date
0,blb,a2f58272-bba6-439d-871e-7a46686ac018,Bloomburrow,2024-08-02 00:00:00+00:00
6,pdft,cc05a3a1-fb38-4d6d-b3d3-032e90b49d00,Aetherdrift Promos,2025-02-14 00:00:00+00:00
7,who,efdfdd99-cee7-4c11-a5e9-03d60f88bdd6,Doctor Who,2023-10-13 00:00:00+00:00
8,woe,79139661-13ee-43c4-8bad-a8c069f1a1df,Wilds of Eldraine,2023-09-08 00:00:00+00:00
11,altc,32dcab1f-9781-4cb5-9ed3-3d1881f906be,Tales of Middle-earth Scene Box,2023-11-04 00:00:00+00:00
...,...,...,...,...
1054,oclb,c0055fd3-8a6e-4f2f-9835-e55447721d09,Battle for Baldur's Gate Oversized Cards,2022-06-10 00:00:00+00:00
1058,smh3,c01b381e-9acc-44bf-82d7-8834a41421fc,Modern Horizons 3 Substitute Cards,2024-06-14 00:00:00+00:00
1061,ovoc,e2b34562-32c8-4221-9fb8-d634b7ad4f86,Crimson Vow Commander Display Commanders,2021-11-19 00:00:00+00:00
1063,tnau,8db8b954-0cf5-4a86-a338-0c8795b557bf,Nauctis: The Sunken Realm Tokens,2027-02-05 00:00:00+00:00


In [2]:
expansions = ['FRA']
formats = ['PremierDraft','TradDraft','PickTwoDraft','Sealed','TradSealed']
name_sync = {
    # 17lands type : (url pattern, gcs folder name)
    "draft": ("draft_data/draft_data_public","draft-data"),
    "game": ("game_data/game_data_public","game-data"),
    "replay": ("replay_data/replay_data_public","replay-data")   
}

In [ ]:
for expansion in expansions:
    for key in name_sync.keys():
        for format in formats:
            # Define the destination path in GCS
            destination_blob_name = f"seventeen-lands/{name_sync[key][1]}/{expansion}/{format}.parquet"
            files_found = gcs.list_files(gcs_client, GCS_BUCKET_NAME, prefix=destination_blob_name)

            # Skip streaming if the file already exists in GCS
            if len(files_found)>0:
                print(f"⏭️ Skipping (already exists): {destination_blob_name}")
                continue
                
            url = f"{DATASETS_PAGE_URL}/{name_sync[key][0]}.{expansion}.{format}.csv.gz"
            print(f"⬇️ Streaming: {url} -> {destination_blob_name}")
            stream.stream_url_to_gcs_parquet(url, gcs_client, GCS_BUCKET_NAME, destination_blob_name)
    


--- Files in bucket 'mtg-open-source-data' ---
⬇️ Streaming: https://17lands-public.s3.amazonaws.com/analysis_data/draft_data/draft_data_public.FRA.PremierDraft.csv.gz -> seventeen-lands/draft-data/FRA/PremierDraft.parquet
Processing URL: https://17lands-public.s3.amazonaws.com/analysis_data/draft_data/draft_data_public.FRA.PremierDraft.csv.gz
Detected file size: 82.06 MB
Detected Format: CSV | Compression: gzip


c:\Dev\mtg\.venv\Lib\site-packages\urllib3\connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host '17lands-public.s3.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Mapped 604 types (601 passed directly to reader).
File is small. Loading all at once...
Uploading Snappy Parquet to gs://mtg-open-source-data/seventeen-lands/draft-data/FRA/PremierDraft.parquet...
✅ Successfully uploaded C:\Users\ARautma\AppData\Local\Temp\tmp0txv41bz.parquet to seventeen-lands/draft-data/FRA/PremierDraft.parquet.
Upload complete!

--- Files in bucket 'mtg-open-source-data' ---
⬇️ Streaming: https://17lands-public.s3.amazonaws.com/analysis_data/draft_data/draft_data_public.FRA.TradDraft.csv.gz -> seventeen-lands/draft-data/FRA/TradDraft.parquet
Processing URL: https://17lands-public.s3.amazonaws.com/analysis_data/draft_data/draft_data_public.FRA.TradDraft.csv.gz


c:\Dev\mtg\.venv\Lib\site-packages\urllib3\connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host '17lands-public.s3.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Detected file size: 6.78 MB
Detected Format: CSV | Compression: gzip
Mapped 604 types (601 passed directly to reader).
File is small. Loading all at once...
Uploading Snappy Parquet to gs://mtg-open-source-data/seventeen-lands/draft-data/FRA/TradDraft.parquet...
✅ Successfully uploaded C:\Users\ARautma\AppData\Local\Temp\tmpjhn0bbec.parquet to seventeen-lands/draft-data/FRA/TradDraft.parquet.
Upload complete!

--- Files in bucket 'mtg-open-source-data' ---
⬇️ Streaming: https://17lands-public.s3.amazonaws.com/analysis_data/draft_data/draft_data_public.FRA.PickTwoDraft.csv.gz -> seventeen-lands/draft-data/FRA/PickTwoDraft.parquet
Processing URL: https://17lands-public.s3.amazonaws.com/analysis_data/draft_data/draft_data_public.FRA.PickTwoDraft.csv.gz


c:\Dev\mtg\.venv\Lib\site-packages\urllib3\connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host '17lands-public.s3.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


... (15 outputs left out)

c:\Dev\mtg\.venv\Lib\site-packages\urllib3\connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host '17lands-public.s3.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Dev\mtg\.venv\Lib\site-packages\urllib3\connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host '17lands-public.s3.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Skipping URL due to HTTP status 403: https://17lands-public.s3.amazonaws.com/analysis_data/replay_data/replay_data_public.FRA.PickTwoDraft.csv.gz

--- Files in bucket 'mtg-open-source-data' ---
⬇️ Streaming: https://17lands-public.s3.amazonaws.com/analysis_data/replay_data/replay_data_public.FRA.Sealed.csv.gz -> seventeen-lands/replay-data/FRA/Sealed.parquet
Processing URL: https://17lands-public.s3.amazonaws.com/analysis_data/replay_data/replay_data_public.FRA.Sealed.csv.gz
Skipping URL due to HTTP status 403: https://17lands-public.s3.amazonaws.com/analysis_data/replay_data/replay_data_public.FRA.Sealed.csv.gz

--- Files in bucket 'mtg-open-source-data' ---
⬇️ Streaming: https://17lands-public.s3.amazonaws.com/analysis_data/replay_data/replay_data_public.FRA.TradSealed.csv.gz -> seventeen-lands/replay-data/FRA/TradSealed.parquet
Processing URL: https://17lands-public.s3.amazonaws.com/analysis_data/replay_data/replay_data_public.FRA.TradSealed.csv.gz
Skipping URL due to HTTP status 40

c:\Dev\mtg\.venv\Lib\site-packages\urllib3\connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host '17lands-public.s3.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


: 

In [5]:
print(t)

None


In [5]:
stream.stream_url_to_gcs_parquet(download_uri, GCS_BUCKET_NAME, "scryfall/default_cards.parquet")

c:\Dev\mtg\.venv\Lib\site-packages\urllib3\connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'data.scryfall.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Detected file size: 77.32 MB
Detected Format: JSONL | Compression: gzip
File is small. Loading all at once...
Uploading Snappy Parquet to gs://mtg-open-source-data/scryfall/default_cards.parquet...
✅ Successfully uploaded C:\Users\ARautma\AppData\Local\Temp\tmpadvp2g6_.parquet to scryfall/default_cards.parquet.
Upload complete!
